In [1]:
import polars as pl
import yaml
from pathlib import Path


with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

In [6]:
VARIANT = "v2"

In [7]:
# join infostop stops with HoWDe home/work/other labels across all 256 hex partitions.
parts = []
for i in range(256):
    stops_lf = pl.scan_parquet(
        f"{config['dirs']['output']}/{config['city']}/stops/{VARIANT}/{i:02x}.parquet"
    )
    howde_lf = pl.scan_parquet(
        f"{config['dirs']['output']}/{config['city']}/howde/{VARIANT}/{i:02x}_howde.parquet"
    )

    part = stops_lf.join(
        howde_lf.select(["useruuid", pl.col("loc").cast(pl.Int32), "location_type"]),
        left_on=["device_id", "stop_id"],
        right_on=["useruuid", "loc"],
        how="left",
    ).collect()
    parts.append(part)

stops = pl.concat(parts)

In [8]:
stops = stops.with_columns(
    pl.col.interval_start.dt.to_string("%Y-%m").alias("year_month")
)

In [9]:
# sort once, then write one parquet per calendar month.
out_dir = Path(
    f"{config['dirs']['output']}/{config['city']}/monthly_partitioned_stops/{VARIANT}_labeled"
)
out_dir.mkdir(exist_ok=True, parents=True)

for year_month, p in (
    stops.sort(["year_month", "device_id", "interval_start"])
    .partition_by("year_month", as_dict=True, maintain_order=True)
    .items()
):
    p.drop("year_month").write_parquet(out_dir / f"{year_month[0]}.parquet")